阶段1：构造模型

在票价确定的前提下，预测各航段的客流量（已完成）。

阶段2：解析过程

根据客流信息分配舱位，以确定总收益。
客舱分为两类：短途类（AB或BC，称为S类）和长途类（AC，称为L类），总容量为C。

根据是否满座进行分类：

若不满座（S+L≤C），则按S、L和C的预测比例分配舱位，以实现收益最大化。
若满座（S+L>C），则需根据销售量变化预测进行取舍，并作如下判断：
若票价AB+BC>AC，则优先将舱位拆分为AB和BC段销售，S的值取min(AB, BC)，其余舱位分配给L。
若票价AB+BC<AC，则优先分配舱位给AC段，剩余舱位分配给AB和BC段。

阶段3：求总收益最大值

以票价为自变量，总收益为因变量，绘制总收益曲线，并求出总收益的最大值。

# 加载资源

In [124]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
import os

def load_resources():
    """
    加载所有预测所需的资源（编码器、模型等）
    
    返回:
    dict: 包含所有加载资源的字典
    """
    resources = {}
    
    # 加载城市标签
    with open('../../my/encoder/city_labels_航班频率加权图标签.json', 'r') as file:
        resources['city_labels'] = json.load(file)
    
    # 加载城市嵌入
    with open('../../my/encoder/城市嵌入编码_航班频率加权图.json', 'r') as file:
        resources['city_embeddings'] = json.load(file)
    
    # 加载城市频率编码
    with open('../../my/encoder/city_map_频率编码.json', 'r') as f:
        resources['city_map'] = json.load(f)
    
    # 加载分类特征编码器
    categorical_columns = ['flt_no', 'aircraft']
    resources['encoders'] = {}
    for col in categorical_columns:
        encoder_path = os.path.join('../../my/encoder/', f"{col}_encoder_all.pkl")
        resources['encoders'][col] = joblib.load(encoder_path)
    
    # 加载标准化器
    resources['scaler_x'] = joblib.load('../../my/encoder/standard_scaler_x.pkl')
    resources['scaler_y'] = joblib.load('../../my/encoder/standard_scaler_y.pkl')
    
    # 加载模型
    resources['model'] = xgb.Booster()
    resources['model'].load_model("../../my/model/频率编码/归一化_xgboost_model_1000.json")
    
    # 预处理城市嵌入为DataFrame格式，方便后续使用
    resources['embedding_df'] = pd.DataFrame.from_dict(
        resources['city_embeddings'], 
        orient='index', 
        columns=['embedding_1', 'embedding_2']
    )
    resources['embedding_df'].index.name = 'city'
    
    # 定义特征列表
    resources['features'] = [
        'flt_no', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 
        'a', 'b', 'c', 'year', 'month', 'day', 'weekday', 'hour', 'minute', 
        'from', 'to', 'unit_price', 'competitor_price', 'a_label', 'b_label', 'c_label', 
        'from_label', 'to_label', 'a_embedding_1', 'a_embedding_2', 'b_embedding_1', 
        'b_embedding_2', 'c_embedding_1', 'c_embedding_2', 'from_embedding_1', 
        'from_embedding_2', 'to_embedding_1', 'to_embedding_2'
    ]
    
    return resources

In [12]:
# # 使用 city_map 替换指定列的值
# columns_to_replace = ['a', 'b', 'c', 'from', 'to']

# # 遍历指定列并直接用 map 映射
# for col in columns_to_replace:
#     data[col] = data[col].map(city_map)

# # 输出结果
# print(data)

               flt_no bd_type    cap aircraft  leg_no  duration  pax  \
0        KgJrsp7Jd78=      窄体  132.0      319       1      1.07   25   
1        P9IRwar34h0=      窄体  189.0      321       1      1.38  151   
2        mJitm0UDfM4=      窄体  132.0      319       1      1.57   38   
3        jXr97M1wpn4=      窄体  132.0      319       1      1.58  109   
4        izjfHOxAho4=      窄体  132.0      319       1      1.80  124   
...               ...     ...    ...      ...     ...       ...  ...   
5999220  BzUm4im0EqA=      窄体  152.0      320       1      3.03   99   
5999221  w+GXbx7u3EM=      窄体  152.0      320       1      2.75  127   
5999222  9ceSbo4suds=      窄体  152.0      320       1      3.48   66   
5999223  +Pv2ewi/JZY=      窄体  158.0      320       1      1.42  157   
5999224  WqHQlgk5y8c=      窄体  158.0      320       1      1.68   95   

                a         b   c  year  month  day  weekday  hour  minute  \
0         29448.0    2515.0 NaN  2023     10    1        6 

# 预测单行数据的客流量

In [125]:
def predict_pax(row_data, resources=None):
    """
    预测单行数据的客流量
    
    参数:
    row_data: DataFrame或Series，包含单行数据
    resources: dict, 可选，包含预加载的资源。如果为None，将自动加载
    
    返回:
    float: 预测的客流量
    """
    # 如果没有提供资源，则加载
    if resources is None:
        resources = load_resources()
    
    # 确保输入是DataFrame格式
    if isinstance(row_data, pd.Series):
        row_data = pd.DataFrame([row_data])
    
    # 创建数据的副本，避免修改原始数据
    row_data = row_data.copy()
    
    # 添加城市标签
    row_data['a_label'] = row_data['a'].map(resources['city_labels'])
    row_data['b_label'] = row_data['b'].map(resources['city_labels'])
    row_data['c_label'] = row_data['c'].map(resources['city_labels'])
    row_data['from_label'] = row_data['from'].map(resources['city_labels'])
    row_data['to_label'] = row_data['to'].map(resources['city_labels'])
    
    # 添加城市嵌入
    cities = ['a', 'b', 'c', 'from', 'to']
    for city in cities:
        # 使用right_index=True方式合并
        row_data = row_data.merge(resources['embedding_df'], left_on=city, right_index=True, how='left')
        
        # 重命名列以避免冲突
        row_data.rename(columns={
            'embedding_1': f'{city}_embedding_1', 
            'embedding_2': f'{city}_embedding_2'
        }, inplace=True)
    
    # 应用频率编码
    for col in ['a', 'b', 'c', 'from', 'to']:
        row_data[col] = row_data[col].map(resources['city_map'])
    
    # 应用标签编码
    for col in resources['encoders'].keys():
        try:
            row_data[col] = resources['encoders'][col].transform(row_data[col])
        except ValueError:
            # 处理未见过的类别
            print(f"警告: {col}列中存在未见过的类别，将使用默认值0")
            row_data[col] = 0
            
    
    # 特征选择
    # print(resources['features'])
    X = row_data[resources['features']]
    pd.set_option('display.max_columns', None)  # 显示所有列
    # print(X)
    
    # 标准化特征
    X_scaled = resources['scaler_x'].transform(X)

    X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
    # print(X_scaled)
    
    # 预测
    dmatrix = xgb.DMatrix(X_scaled)
    # print(dmatrix)
    scaled_predictions = resources['model'].predict(dmatrix)
    
    # 反标准化预测结果
    predictions = resources['scaler_y'].inverse_transform(scaled_predictions.reshape(-1, 1))
    
    return predictions[0][0]

In [106]:
# data = pd.read_csv('./pre_2023-2024_with_comp_test.csv', dtype={'flt_no': str})
# print(data.info())
# # 单行预测示例
# test_row = data.iloc[[100]]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 282755 entries, 0 to 282754
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   flt_no            282755 non-null  object 
 1   cap               282755 non-null  int64  
 2   aircraft          282755 non-null  object 
 3   legs              282755 non-null  int64  
 4   leg_no            282755 non-null  int64  
 5   duration          282727 non-null  float64
 6   pax               282755 non-null  int64  
 7   a                 282755 non-null  object 
 8   b                 282755 non-null  object 
 9   c                 112308 non-null  object 
 10  unit_price        282755 non-null  float64
 11  competitor_price  282755 non-null  float64
 12  year              282755 non-null  int64  
 13  month             282755 non-null  int64  
 14  day               282755 non-null  int64  
 15  weekday           282755 non-null  int64  
 16  hour              28

In [42]:
# test_row
# # 简洁版本
# print("test_row 的数据类型和值:")
# for column, value in test_row.items():
#     print(f"{column}: {value} (类型: {type(value).__name__})")

test_row 的数据类型和值:
flt_no: 0    7558
Name: flt_no, dtype: object (类型: Series)
cap: 0    110
Name: cap, dtype: int64 (类型: Series)
aircraft: 0    195
Name: aircraft, dtype: object (类型: Series)
legs: 0    1
Name: legs, dtype: int64 (类型: Series)
leg_no: 0    1
Name: leg_no, dtype: int64 (类型: Series)
duration: 0    1.3
Name: duration, dtype: float64 (类型: Series)
pax: 0    97
Name: pax, dtype: int64 (类型: Series)
a: 0    AAT
Name: a, dtype: object (类型: Series)
b: 0    URC
Name: b, dtype: object (类型: Series)
c: 0    NaN
Name: c, dtype: object (类型: Series)
unit_price: 0    470.474227
Name: unit_price, dtype: float64 (类型: Series)
competitor_price: 0   -68.7985
Name: competitor_price, dtype: float64 (类型: Series)
year: 0    2023
Name: year, dtype: int64 (类型: Series)
month: 0    1
Name: month, dtype: int64 (类型: Series)
day: 0    1
Name: day, dtype: int64 (类型: Series)
weekday: 0    6
Name: weekday, dtype: int64 (类型: Series)
hour: 0    14
Name: hour, dtype: int64 (类型: Series)
minute: 0    35
Name: min

In [107]:
# predicted_pax = predict_pax(test_row)

In [108]:
# predicted_pax

152.88757

# 一行航线数据拆分为 3 个航段数据

In [126]:
import pandas as pd

def split_flight_row(row: pd.Series) -> list:
    """
    将 merged_flights.csv 中的一行航线数据拆分为 3 个航段数据。

    参数:
        row (pd.Series): 航线数据的一行

    返回:
        list: 包含 3 个航段数据的列表，每个航段数据为一个 pandas.DataFrame
    """
    # 定义需要拆分的列
    suffix_columns = ['cap', 'duration', 'pax', 'unit_price', 'competitor_price', 'hour', 'minute']
    keep_columns = ['flt_no', 'aircraft', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday']
    leg_suffix_map = {1: 'ab', 2: 'bc', 3: 'ac'}

    # 目标字段排列顺序
    column_order = [
        'flt_no', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'pax', 'a', 'b', 'c', 
        'unit_price', 'competitor_price', 'year', 'month', 'day', 'weekday', 'hour', 'minute', 
        'from', 'to'
    ]

    # 存储拆分后的航段数据
    result = []

    # 遍历 leg_no 进行拆分
    for leg_no, suffix in leg_suffix_map.items():
        new_row = {col: row[col] for col in keep_columns}  # 复制不变的列
        new_row['leg_no'] = leg_no  # 添加航段编号
        new_row['legs'] = 3  # 所有航段的 legs 值设为 3
        
        # 更新航段的起始和终点
        if leg_no == 1:
            new_row['from'] = row['a']
            new_row['to'] = row['b']
        elif leg_no == 2:
            new_row['from'] = row['b']
            new_row['to'] = row['c']
        elif leg_no == 3:
            new_row['from'] = row['a']
            new_row['to'] = row['c']

        # 拆分有后缀的列
        for col in suffix_columns:
            new_col_name = f"{col}_{suffix}"
            new_row[col] = row[new_col_name] if new_col_name in row else None

        # 转换为 DataFrame 并重新排序列
        segment_df = pd.DataFrame([new_row])
        segment_df = segment_df[column_order]  # 按照目标顺序排序

        result.append(segment_df)

    # print(result)
    return result

# # 读取 merged_flights.csv 并测试
# file_path = "./merged_flights.csv"

# if os.path.exists(file_path):
#     df_merged = pd.read_csv(file_path)
#     sample_row = df_merged.iloc[3]  # 选取第一行进行测试
#     split_segments_list = split_flight_row(sample_row)

#     # 展示拆分后的数据

#     # print(split_segments_list)

#     # for i, segment_df in enumerate(split_segments_list, start=1):
#     #     # 展示拆分后的数据
#     #     print(segment_df.head())
#     #     # print(segment_df.info())

# else:
#     print(f"❌ 错误: 文件 {file_path} 未找到，请检查文件路径！")


In [85]:
# pd.read_csv("./route_price_stats.csv").head()

,from,to,平均价格,价格标准差,航班数量
0,ENY,KRL,2900.00,290.000,1
1,DNH,PVG,2860.22,286.022,1
2,PVG,DNH,2860.00,286.000,1
3,URC,NGQ,2533.65,70.830,211
4,NGQ,URC,2529.94,63.180,226


# 预测输入的一行航线数据的 3 个航段的客流量

In [128]:
import pandas as pd

def predict_flight_pax(row: pd.Series, resources=None) -> list:
    """
    预测输入的航线数据的 3 个航段的客流量。

    参数:
        row (pd.Series): 航线数据的一行
        resources (dict, optional): 预加载的资源。如果为None，将自动加载

    返回:
        list: 预测的 3 个航段客流量 [pax_ab, pax_bc, pax_ac]
    """
    # 如果没有提供资源，则加载
    if resources is None:
        resources = load_resources()
    
    # 1. 拆分航线数据为 3 个航段
    segments = split_flight_row(row)  # 拆分出的航段数据，每个是 DataFrame

    # 2. 预测 3 个航段的客流量
    pax_predictions = []
    for segment_df in segments:
        predicted_pax = predict_pax(segment_df, resources)  # 传入预加载的资源
        pax_predictions.append(predicted_pax)

    return pax_predictions

# # 示例测试：
# file_path = "./merged_flights.csv"

# if os.path.exists(file_path):
#     df_merged = pd.read_csv(file_path)
    
#     # 选取一行航线数据进行测试
#     sample_row = df_merged.iloc[460]  
#     predicted_pax_list = predict_flight_pax(sample_row)

#     # 输出预测结果
#     print("预测的航段客流量:", predicted_pax_list)

# else:
#     print(f"❌ 错误: 文件 {file_path} 未找到，请检查文件路径！")


预测的航段客流量: [86.33025, 84.689995, 57.6121]


# 根据预测客流和票价进行舱位分配

In [129]:
def allocate_seats(pax_predictions, ab_price, bc_price, ac_price, cap):
    """
    根据收益最大化策略，分配航段 AB, BC, AC 的客舱容量。
    
    输入:
    - pax_predictions: 字典，包含 AB, BC, AC 的乘客数预测值，例如 {'AB_PAX': 120, 'BC_PAX': 110, 'AC_PAX': 130}。
    - ab_price: float，航段 AB 的票价。
    - bc_price: float，航段 BC 的票价。
    - ac_price: float，航段 AC 的票价。
    - cap: int，总客舱容量。

    输出:
    - tuple: (allocation, revenue)
        - allocation: 字典，包含分配给 AB, BC, AC 的舱位数量
        - revenue: float，预计总收入
    """
    # 获取航段预测乘客数
    ab_pax = round(pax_predictions['AB_PAX'])
    bc_pax = round(pax_predictions['BC_PAX'])
    ac_pax = round(pax_predictions['AC_PAX'])

    # 短途类（S 类）的最大容量
    s_max = max(ab_pax, bc_pax)
    s_min = min(ab_pax, bc_pax)

    # 分配结果初始化
    allocation = {'AB': 0, 'BC': 0, 'AC': 0}

    # 判断是否满座
    if s_max + ac_pax <= cap:
        # 未满座：按比例分配 S 类和 L 类
        total_pax = s_max + ac_pax
        s_cap = round(cap * (s_max / total_pax))  # S 类分配的容量
        l_cap = cap - s_cap  # L 类分配的容量

        # S 类：直接分配 s_cap（AB 和 BC 同时销售）
        allocation['AB'] = s_cap
        allocation['BC'] = s_cap

        # L 类：分配剩余容量
        allocation['AC'] = l_cap
    else:
        # 已满座：根据收益优先级分配
        s_revenue = ab_price + bc_price
        l_revenue = ac_price

        if s_revenue > l_revenue:
            # 优先分配给 S 类的 min(AB_PAX, BC_PAX)
            s_cap = min(s_min, cap)
            allocation['AB'] = s_cap
            allocation['BC'] = s_cap

            # 剩余容量优先分配给 L 类
            remaining_cap = cap - s_cap
            allocation['AC'] = min(ac_pax, remaining_cap)

            # 如果 L 类分配后还有剩余容量,直接分配给 S 类
            remaining_after_l = remaining_cap - allocation['AC']
            if remaining_after_l > 0:
                allocation['AB'] += remaining_after_l
                allocation['BC'] += remaining_after_l
        else:
            # 优先分配给 L 类
            allocation['AC'] = min(ac_pax, cap)  # L 类尽量满足 AC 的预测乘客数
            remaining_cap = cap - allocation['AC']  # 剩余容量分配给 S 类
            allocation['AB'] = remaining_cap
            allocation['BC'] = remaining_cap

    # 计算预期收入
    revenue = (min(ab_pax, allocation['AB']) * ab_price + 
              min(bc_pax, allocation['BC']) * bc_price + 
              min(ac_pax, allocation['AC']) * ac_price)

    return allocation, revenue


# 根据一行航线数据进行舱位分配

In [130]:
import pandas as pd

def allocate_flight_seats(row: pd.Series, resources=None) -> tuple:
    """
    根据航线数据进行舱位分配。
    1. 预测 3 个航段的客流量。
    2. 使用收益最大化策略，分配航段 AB, BC, AC 的客舱容量。

    参数:
        row (pd.Series): 航线数据的一行
        resources (dict, optional): 预加载的资源。如果为None，将自动加载

    返回:
        tuple: (allocation, revenue, pax_predictions)
            - allocation: 字典，包含分配给 AB, BC, AC 的舱位数量
            - revenue: float，预计总收入
            - pax_predictions: 预测的 3 个航段客流量
    """
    # 如果没有提供资源，则加载
    if resources is None:
        resources = load_resources()
    
    # 1. 预测航段客流量
    pax_predictions = predict_flight_pax(row, resources)  # 传入预加载的资源

    # 2. 获取航段票价和客舱容量
    ab_price = row['unit_price_ab']
    bc_price = row['unit_price_bc']
    ac_price = row['unit_price_ac']
    cap = row['cap_ab']  # 总客舱容量

    # 3. 预测座位分配
    allocation, revenue = allocate_seats(
        pax_predictions={
            'AB_PAX': pax_predictions[0],
            'BC_PAX': pax_predictions[1],
            'AC_PAX': pax_predictions[2],
        },
        ab_price=ab_price,
        bc_price=bc_price,
        ac_price=ac_price,
        cap=cap
    )

    return allocation, revenue, pax_predictions

# # 读取 merged_flights.csv 并测试
# file_path = "./merged_flights.csv"

# if os.path.exists(file_path):
#     df_merged = pd.read_csv(file_path)

#     # 选取一行航线数据进行测试
#     sample_row = df_merged.iloc[460]
#     allocation_result, expected_revenue, pax_predictions = allocate_flight_seats(sample_row)

#     # 输出结果
#     print("座位分配结果:", allocation_result)
#     print("预计总收入:", expected_revenue)
#     print("预测的航段客流量:", pax_predictions)

# else:
#     print(f"❌ 错误: 文件 {file_path} 未找到，请检查文件路径！")


座位分配结果: {'AB': 96, 'BC': 96, 'AC': 65}
预计总收入: 164113.68207892735
预测的航段客流量: [86.33025, 84.689995, 57.6121]


# 预测一趟航线的票价

In [135]:
import pandas as pd
import numpy as np
from itertools import product

def get_price_stats(from_city, to_city, price_stats):
    """
    根据 `from` 和 `to` 获取航段的价格均值和标准差。
    
    参数:
    - from_city: str, 出发地
    - to_city: str, 目的地
    - price_stats: DataFrame, 航段价格统计信息

    返回:
    - (float, float): (均值, 标准差)
    """
    match = price_stats[(price_stats['from'] == from_city) & (price_stats['to'] == to_city)]
    if not match.empty:
        return match.iloc[0]['平均价格'], match.iloc[0]['价格标准差']
    else:
        return None, None  # 如果找不到匹配项，返回 None

def get_candidate_prices(mean_price, std_price):
    """
    生成候选票价，在均值 ± 3 标准差的范围内。
    
    参数:
    - mean_price: float, 票价均值
    - std_price: float, 票价标准差

    返回:
    - list: 可能的票价列表
    """
    if mean_price is None or std_price is None:
        return []  # 若无数据，则返回空列表
    return np.linspace(mean_price - 3 * std_price, mean_price + 3 * std_price, num=7).tolist()

def optimize_ticket_prices(row: pd.Series, price_stats: pd.DataFrame, resources=None) -> dict:
    """
    优化票价以获得最大收入。

    参数:
    - row: pd.Series, 航线数据的一行
    - price_stats: pd.DataFrame, 航线价格统计信息
    - resources: dict, 可选，包含预加载的资源。如果为None，将自动加载

    返回:
    - dict: 包含最优票价组合、预测客流、座位分配和预期收入
    """
    # 如果没有提供资源，则加载
    if resources is None:
        resources = load_resources()
        
    # 1. 获取各航段的价格统计信息
    ab_mean, ab_std = get_price_stats(row['a'], row['b'], price_stats)
    bc_mean, bc_std = get_price_stats(row['b'], row['c'], price_stats)
    ac_mean, ac_std = get_price_stats(row['a'], row['c'], price_stats)

    # 2. 生成候选票价（在均值 ± 3σ 之间）
    ab_prices = get_candidate_prices(ab_mean, ab_std)
    bc_prices = get_candidate_prices(bc_mean, bc_std)
    ac_prices = get_candidate_prices(ac_mean, ac_std)

    # 3. 获取航线的总舱位容量
    cap = row['cap_ab']

    # 4. 存储最优解
    best_revenue = 0
    best_allocation = None
    best_prices = None
    best_predictions = None

    # 5. 遍历所有可能的票价组合
    for ab_p, bc_p, ac_p in product(ab_prices, bc_prices, ac_prices):
        if ab_p >= ac_p or bc_p >= ac_p:  # 票价限制，确保长航段票价不低于短航段
            continue

        # 更新当前 row 中的票价
        row_copy = row.copy()  # 创建副本避免修改原始数据
        row_copy['unit_price_ab'] = ab_p
        row_copy['unit_price_bc'] = bc_p
        row_copy['unit_price_ac'] = ac_p

        # 6. 预测 3 个航段的客流量
        pax_predictions = predict_flight_pax(row_copy, resources)  # 传入预加载的资源

        # 7. 计算座位分配
        allocation, revenue = allocate_seats(
            pax_predictions={
                'AB_PAX': pax_predictions[0],
                'BC_PAX': pax_predictions[1],
                'AC_PAX': pax_predictions[2],
            },
            ab_price=ab_p,
            bc_price=bc_p,
            ac_price=ac_p,
            cap=cap
        )

        # 8. 更新最优票价
        if revenue > best_revenue:
            best_revenue = revenue
            best_allocation = allocation
            best_prices = {'AB': ab_p, 'BC': bc_p, 'AC': ac_p}
            best_predictions = pax_predictions

    # 9. 返回最优票价组合
    return {
        'optimal_prices': best_prices,
        'predicted_pax': best_predictions,
        'seat_allocation': best_allocation,
        'expected_revenue': best_revenue
    }

# # 读取 route_price_stats.csv 并测试
# price_stats_path = "./route_price_stats.csv"

# if os.path.exists(price_stats_path):
#     df_price_stats = pd.read_csv(price_stats_path)

#     # 读取 merged_flights.csv 并选取一行进行测试
#     file_path = "./merged_flights.csv"
#     if os.path.exists(file_path):
#         df_merged = pd.read_csv(file_path)
#         sample_row = df_merged.loc[460].copy()  # 选取测试行
#         resources = load_resources()
#         optimal_result = optimize_ticket_prices(sample_row, df_price_stats,resources)


#     else:
#         print(f"❌ 错误: 文件 {file_path} 未找到，请检查文件路径！")

# else:
#     print(f"❌ 错误: 文件 {price_stats_path} 未找到，请检查文件路径！")


In [136]:
# sample_row = df_merged.loc[460].copy()  # 选取测试行
# resources = load_resources()
# optimal_result = optimize_ticket_prices(sample_row, df_price_stats,resources)

In [137]:
# optimal_result

{'optimal_prices': {'AB': 1346.3899999999999, 'BC': 732.37, 'AC': 1614.87},
 'predicted_pax': [89.90592, 86.50165, 65.13119],
 'seat_allocation': {'AB': 93, 'BC': 93, 'AC': 68},
 'expected_revenue': 289857.83999999997}

# 批量预测票价

In [138]:
# def batch_optimize_ticket_prices(df: pd.DataFrame, price_stats: pd.DataFrame) -> pd.DataFrame:
#     """
#     批量优化多条航线的票价。
    
#     参数:
#     - df: pd.DataFrame, 包含多条航线数据的DataFrame
#     - price_stats: pd.DataFrame, 航线价格统计信息
    
#     返回:
#     - pd.DataFrame: 包含原始数据和优化结果的DataFrame
#     """
#     # 加载资源（只加载一次）
#     resources = load_resources()
    
#     # 存储优化结果
#     results = []
    
#     # 显示进度条
#     from tqdm import tqdm
    
#     # 对每一行数据进行优化
#     for idx, row in tqdm(df.iterrows(), total=len(df), desc="优化票价"):
#         try:
#             # 优化票价
#             optimization_result = optimize_ticket_prices(row, price_stats, resources)
            
#             # 提取优化结果
#             optimal_prices = optimization_result['optimal_prices']
#             predicted_pax = optimization_result['predicted_pax']
#             seat_allocation = optimization_result['seat_allocation']
#             expected_revenue = optimization_result['expected_revenue']
            
#             # 创建结果字典
#             result = {
#                 # 保留原始数据的索引和关键信息
#                 'index': idx,
#                 'year': row.get('year', ''),
#                 'month': row.get('month', ''),
#                 'day': row.get('day', ''),
#                 'flt_no': row.get('flt_no', ''),
#                 'a': row.get('a', ''),
#                 'b': row.get('b', ''),
#                 'c': row.get('c', ''),
#                 'cap_ab': row.get('cap_ab', 0),
                
#                 # 添加优化后的票价
#                 'optimal_price_ab': optimal_prices.get('AB', 0) if optimal_prices else 0,
#                 'optimal_price_bc': optimal_prices.get('BC', 0) if optimal_prices else 0,
#                 'optimal_price_ac': optimal_prices.get('AC', 0) if optimal_prices else 0,
                
#                 # 添加预测的客流量
#                 'predicted_pax_ab': predicted_pax[0] if predicted_pax else 0,
#                 'predicted_pax_bc': predicted_pax[1] if predicted_pax else 0,
#                 'predicted_pax_ac': predicted_pax[2] if predicted_pax else 0,
                
#                 # 添加座位分配
#                 'allocated_seats_ab': seat_allocation.get('AB', 0) if seat_allocation else 0,
#                 'allocated_seats_bc': seat_allocation.get('BC', 0) if seat_allocation else 0,
#                 'allocated_seats_ac': seat_allocation.get('AC', 0) if seat_allocation else 0,
                
#                 # 添加预期收入
#                 'expected_revenue': expected_revenue,
                
#                 # 添加原始票价（用于比较）
#                 'original_price_ab': row.get('unit_price_ab', 0),
#                 'original_price_bc': row.get('unit_price_bc', 0),
#                 'original_price_ac': row.get('unit_price_ac', 0),
                
#                 # 添加原始客流量（用于比较）
#                 'original_pax_ab': row.get('pax_ab', 0),
#                 'original_pax_bc': row.get('pax_bc', 0),
#                 'original_pax_ac': row.get('pax_ac', 0),
                
#                 # 计算原始收入（用于比较）
#                 'original_revenue': (
#                     row.get('pax_ab', 0) * row.get('unit_price_ab', 0) +
#                     row.get('pax_bc', 0) * row.get('unit_price_bc', 0) +
#                     row.get('pax_ac', 0) * row.get('unit_price_ac', 0)
#                 ),
                
#                 # 计算收入提升百分比
#                 'revenue_improvement_pct': 0  # 将在后面计算
#             }
            
#             # 计算收入提升百分比
#             original_revenue = result['original_revenue']
#             if original_revenue > 0:
#                 result['revenue_improvement_pct'] = (
#                     (expected_revenue - original_revenue) / original_revenue * 100
#                 )
            
#             results.append(result)
            
#         except Exception as e:
#             print(f"处理索引 {idx} 的行时出错: {str(e)}")
#             # 可以选择添加一个空结果或跳过
#             results.append({
#                 'index': idx,
#                 'error': str(e)
#             })
    
#     # 转换为DataFrame
#     results_df = pd.DataFrame(results)
    
#     return results_df

# # # 使用示例
# # df_merged = pd.read_csv('./merged_flights.csv')
# # price_stats = pd.read_csv('./route_price_stats.csv')

# # # 选取一部分数据进行测试（例如前100行）
# # sample_df = df_merged.head(10)

# # # 批量优化票价
# # optimization_results = batch_optimize_ticket_prices(sample_df, price_stats)

# # # 保存结果
# # optimization_results.to_csv('ticket_price_optimization_results.csv', index=False)

# # # 显示优化结果摘要
# # print(f"平均收入提升: {optimization_results['revenue_improvement_pct'].mean():.2f}%")
# # print(f"最大收入提升: {optimization_results['revenue_improvement_pct'].max():.2f}%")

优化票价: 100%|█████████████████████████████████| 10/10 [04:25<00:00, 26.53s/it]

平均收入提升: 65.69%
最大收入提升: 116.48%


In [ ]:
def process_single_flight(row, price_stats, resources=None):
    """处理单个航班的票价优化任务"""
    try:
        # 优化票价
        optimization_result = optimize_ticket_prices(row, price_stats, resources)
        
        # 提取优化结果
        optimal_prices = optimization_result['optimal_prices']
        predicted_pax = optimization_result['predicted_pax']
        seat_allocation = optimization_result['seat_allocation']
        expected_revenue = optimization_result['expected_revenue']
        
        # 计算原始收入
        original_revenue = (
            row.get('pax_ab', 0) * row.get('unit_price_ab', 0) +
            row.get('pax_bc', 0) * row.get('unit_price_bc', 0) +
            row.get('pax_ac', 0) * row.get('unit_price_ac', 0)
        )
        
        # 计算收入提升
        revenue_improvement = 0
        revenue_improvement_amount = 0
        if original_revenue > 0:
            revenue_improvement = (expected_revenue - original_revenue) / original_revenue * 100
            revenue_improvement_amount = expected_revenue - original_revenue
        
        return {
            # 航班基本信息
            'flt_no': row.get('flt_no', ''),
            'year': row.get('year', ''),
            'month': row.get('month', ''),
            'day': row.get('day', ''),
            'a': row.get('a', ''),
            'b': row.get('b', ''),
            'c': row.get('c', ''),
            
            # 原始价格
            'original_ab_price': row.get('unit_price_ab', 0),
            'original_bc_price': row.get('unit_price_bc', 0),
            'original_ac_price': row.get('unit_price_ac', 0),
            
            # 原始客流量
            'original_ab_pax': row.get('pax_ab', 0),
            'original_bc_pax': row.get('pax_bc', 0),
            'original_ac_pax': row.get('pax_ac', 0),
            
            # 原始收入
            'original_revenue': original_revenue,
            
            # 优化后的价格
            'optimal_ab_price': optimal_prices.get('AB', 0) if optimal_prices else 0,
            'optimal_bc_price': optimal_prices.get('BC', 0) if optimal_prices else 0,
            'optimal_ac_price': optimal_prices.get('AC', 0) if optimal_prices else 0,
            
            # 优化后的客流预测
            'predicted_ab_pax': predicted_pax[0] if predicted_pax else 0,
            'predicted_bc_pax': predicted_pax[1] if predicted_pax else 0,
            'predicted_ac_pax': predicted_pax[2] if predicted_pax else 0,
            
            # 优化后的座位分配
            'allocated_ab_seats': seat_allocation.get('AB', 0) if seat_allocation else 0,
            'allocated_bc_seats': seat_allocation.get('BC', 0) if seat_allocation else 0,
            'allocated_ac_seats': seat_allocation.get('AC', 0) if seat_allocation else 0,
            
            # 收入和提升
            'expected_revenue': expected_revenue,
            'revenue_improvement': revenue_improvement,
            'revenue_improvement_amount': revenue_improvement_amount
        }
    except Exception as e:
        print(f"处理航班 {row.get('flt_no', '未知')} 时出错: {str(e)}")
        return None

def batch_optimize_ticket_prices_parallel(df: pd.DataFrame, price_stats: pd.DataFrame) -> pd.DataFrame:
    """
    批量优化多条航线的票价，使用多进程加速。
    
    参数:
    - df: pd.DataFrame, 包含多条航线数据的DataFrame
    - price_stats: pd.DataFrame, 航线价格统计信息
    
    返回:
    - pd.DataFrame: 包含原始数据和优化结果的DataFrame
    """
    from multiprocessing import Pool, cpu_count
    import time
    from tqdm import tqdm
    
    # 记录开始时间
    start_time = time.time()
    print(f"开始运行时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # 加载资源（只加载一次）
    resources = load_resources()
    
    # 设置进程数
    n_processes = max(1, cpu_count() - 1)
    print(f"使用 {n_processes} 个CPU核心进行并行处理")
    
    # 创建进程池并执行
    with Pool(processes=n_processes) as pool:
        results = list(tqdm(
            pool.starmap(
                process_single_flight,
                [(row, price_stats, resources) for _, row in df.iterrows()]
            ),
            total=len(df),
            desc="优化票价"
        ))
    
    # 过滤掉None结果
    results = [r for r in results if r is not None]
    
    # 转换为DataFrame
    results_df = pd.DataFrame(results)
    
    # 计算统计信息
    total_flights = len(results_df)
    improved_flights = len(results_df[results_df['revenue_improvement'] > 0])
    improvement_ratio = improved_flights / total_flights * 100
    
    # 输出统计信息
    print("\n优化结果统计：")
    print(f"处理的航班数量: {total_flights}")
    print(f"收入提升的航班数量: {improved_flights}")
    print(f"收入提升航班占比: {improvement_ratio:.2f}%")
    print(f"\n收入提升详情:")
    print(f"平均收入提升: {results_df['revenue_improvement'].mean():.2f}%")
    print(f"最大收入提升: {results_df['revenue_improvement'].max():.2f}%")
    print(f"最小收入提升: {results_df['revenue_improvement'].min():.2f}%")
    print(f"总收入提升金额: {results_df['revenue_improvement_amount'].sum():,.2f}")
    
    # 添加分布统计
    print("\n收入提升分布:")
    improvement_ranges = [
        (results_df['revenue_improvement'] <= 0).sum(),
        ((results_df['revenue_improvement'] > 0) & (results_df['revenue_improvement'] <= 10)).sum(),
        ((results_df['revenue_improvement'] > 10) & (results_df['revenue_improvement'] <= 50)).sum(),
        ((results_df['revenue_improvement'] > 50) & (results_df['revenue_improvement'] <= 100)).sum(),
        (results_df['revenue_improvement'] > 100).sum()
    ]
    
    print(f"无提升航班数: {improvement_ranges[0]} ({improvement_ranges[0]/total_flights*100:.2f}%)")
    print(f"提升0-10%航班数: {improvement_ranges[1]} ({improvement_ranges[1]/total_flights*100:.2f}%)")
    print(f"提升10-50%航班数: {improvement_ranges[2]} ({improvement_ranges[2]/total_flights*100:.2f}%)")
    print(f"提升50-100%航班数: {improvement_ranges[3]} ({improvement_ranges[3]/total_flights*100:.2f}%)")
    print(f"提升>100%航班数: {improvement_ranges[4]} ({improvement_ranges[4]/total_flights*100:.2f}%)")
    
    # 保存结果
    results_df.to_csv('parallel_ticket_price_optimization_results.csv', index=False)
    
    # 记录结束时间
    end_time = time.time()
    print(f"\n结束运行时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"总运行时长: {(end_time - start_time) / 60:.2f} 分钟")
    
    return results_df

# 使用示例
if __name__ == "__main__":
    # 读取数据
    df_merged = pd.read_csv('./merged_flights.csv')
    price_stats = pd.read_csv('./route_price_stats.csv')
    
    # 可以选择只处理部分数据进行测试
    sample_df = df_merged.sample(n=10, random_state=42)
    
    # 批量优化票价
    optimization_results = batch_optimize_ticket_prices_parallel(sample_df, price_stats)

开始运行时间: 2025-03-17 17:19:29
使用 15 个CPU核心进行并行处理
